# FMP Macro

Read-first examples for FMP-backed macroeconomic series, treasury data, yield curve history, calendars, and risk premium snapshots.

In [1]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openbb import obb

for env_dir in [Path.cwd(), *Path.cwd().parents]:
    env_path = env_dir / ".env"
    if env_path.exists():
        load_dotenv(env_path, override=False)
        break

from quant_warehouse.ingest.credentials import configure_openbb_credentials
from quant_warehouse.ingest.openbb_fetch import SECTION_ROUTES
from quant_warehouse.platforms.data_providers.fmp.sections import (
    FMP_EXTENDED_EQUITY_SECTIONS,
    FMP_HISTORICAL_EQUITY_SECTIONS,
    FMP_HISTORICAL_ETF_SECTIONS,
)
from quant_warehouse.warehouse import Warehouse
from quant_warehouse.warehouse.sections import (
    CRYPTO_PRICES_LIBRARY,
    CRYPTO_PRICES_SECTION,
    CURRENCY_PRICES_LIBRARY,
    CURRENCY_PRICES_SECTION,
    DEFAULT_CRYPTO_SYMBOLS,
    DEFAULT_CURRENCY_SYMBOLS,
    DEFAULT_ECONOMIC_SERIES,
    DEFAULT_INDEX_SYMBOLS,
    EQUITY_CALENDAR_SECTIONS,
    EQUITY_PRICES_LIBRARY,
    ETF_PRICES_LIBRARY,
    ETF_PRICES_SECTION,
    FUND_PRICES_LIBRARY,
    FUND_PRICES_SECTION,
    INDEX_PRICES_LIBRARY,
    INDEX_PRICES_SECTION,
    MACRO_CALENDAR_LIBRARY,
    MACRO_CALENDAR_SECTION,
    MACRO_CALENDAR_SYMBOL,
    MACRO_ECONOMIC_LIBRARY,
    MACRO_ECONOMIC_SECTION,
    MACRO_RISK_PREMIUM_LIBRARY,
    MACRO_RISK_PREMIUM_SECTION,
    MACRO_TREASURY_LIBRARY,
    MACRO_TREASURY_SECTION,
    MACRO_YIELD_CURVE_LIBRARY,
    MACRO_YIELD_CURVE_SECTION,
    RISK_PREMIUM_SYMBOL,
    TREASURY_BUNDLE_SYMBOL,
    YIELD_CURVE_BUNDLE_SYMBOL,
    fundamental_library,
)
from quant_warehouse.warehouse.storage import provider_library

configure_openbb_credentials()

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

warehouse = Warehouse()
PROVIDER = "fmp"
RUN_REFRESH = False

In [2]:
def preview(frame: pd.DataFrame, rows: int = 5) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()
    return frame.tail(rows)


def state(symbol: str, section: str, provider: str = PROVIDER) -> dict[str, object] | None:
    item = warehouse.catalog.get(symbol=symbol, section=section, provider=provider)
    return asdict(item) if item is not None else None


def state_table(symbol: str, sections: list[str] | tuple[str, ...], provider: str = PROVIDER) -> pd.DataFrame:
    rows = [state(symbol, section, provider=provider) for section in sections]
    rows = [row for row in rows if row is not None]
    table = pd.DataFrame(rows)
    if not table.empty and "columns_present" in table.columns:
        table.insert(
            table.columns.get_loc("columns_present"),
            "column_count",
            table["columns_present"].map(len),
        )
    return table


def route_table(section_to_base_library: dict[str, str]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "section": section,
                "openbb_route": SECTION_ROUTES.get(section),
                "provider_library": provider_library(base_library, PROVIDER),
            }
            for section, base_library in section_to_base_library.items()
        ]
    )


def run_openbb_route(label: str, fn):
    try:
        result = fn()
        frame = result.to_df()
        return {
            "label": label,
            "provider": getattr(result, "provider", None),
            "rows": 0 if frame is None else len(frame),
            "columns": [] if frame is None else list(frame.columns),
            "error": None,
        }
    except Exception as exc:
        return {
            "label": label,
            "provider": PROVIDER,
            "rows": None,
            "columns": [],
            "error": str(exc),
        }

## Available Macro Sections

In [3]:
macro_route_libraries = {
    MACRO_ECONOMIC_SECTION: MACRO_ECONOMIC_LIBRARY,
    MACRO_TREASURY_SECTION: MACRO_TREASURY_LIBRARY,
    MACRO_YIELD_CURVE_SECTION: MACRO_YIELD_CURVE_LIBRARY,
    MACRO_CALENDAR_SECTION: MACRO_CALENDAR_LIBRARY,
    MACRO_RISK_PREMIUM_SECTION: MACRO_RISK_PREMIUM_LIBRARY,
}
route_table(macro_route_libraries)

,section,openbb_route,provider_library
0,macro_economic,None,fmp_macro_economic
1,macro_treasury,None,fmp_macro_treasury
2,macro_yield_curve,None,fmp_macro_yield_curve
3,macro_calendar,None,fmp_macro_calendar
4,macro_risk_premium,None,fmp_macro_risk_premium


In [4]:
pd.Series(DEFAULT_ECONOMIC_SERIES, name="default_economic_series")

0                                               GDP
1                                               CPI
2                                  unemploymentRate
3                                     inflationRate
4                                      federalFunds
5                                       retailSales
6                    industrialProductionTotalIndex
7                               totalNonfarmPayroll
8                                     initialClaims
9    newPrivatelyOwnedHousingUnitsStartedTotalUnits
Name: default_economic_series, dtype: object

## Refresh Controls

In [5]:
if RUN_REFRESH:
    warehouse.refresh_macro(
        economic_series=["GDP", "CPI"],
        include_treasury_rates=True,
        include_yield_curve=True,
        include_calendar=True,
        include_risk_premium=True,
        provider=PROVIDER,
    )

## Storage Coverage

In [6]:
macro_states = [
    state("GDP", MACRO_ECONOMIC_SECTION),
    state("CPI", MACRO_ECONOMIC_SECTION),
    state(TREASURY_BUNDLE_SYMBOL, MACRO_TREASURY_SECTION),
    state(YIELD_CURVE_BUNDLE_SYMBOL, MACRO_YIELD_CURVE_SECTION),
    state(MACRO_CALENDAR_SYMBOL, MACRO_CALENDAR_SECTION),
    state(RISK_PREMIUM_SYMBOL, MACRO_RISK_PREMIUM_SECTION),
]
pd.DataFrame([row for row in macro_states if row is not None])

,symbol,section,provider,min_date,max_date,row_count,columns_present,last_fetched_at
0,GDP,macro_economic,fmp,1947-01-01,2026-01-01,317,"(value,)",2026-06-25T13:32:56.386481+00:00
1,CPI,macro_economic,fmp,1947-01-01,2026-05-01,952,"(value,)",2026-06-25T13:32:56.846162+00:00
2,TREASURY_CURVE,macro_treasury,fmp,1990-01-02,2026-06-24,9127,"(macro__ust_month1, macro__ust_month2, macro__...",2026-06-25T13:33:07.926183+00:00
3,YIELD_CURVE,macro_yield_curve,fmp,2005-01-03,2026-06-15,1119,"(macro__yc_month1, macro__yc_month2, macro__yc...",2026-06-20T16:09:47.877403+00:00
4,MACRO_CALENDAR,macro_calendar,fmp,2013-01-02,2026-06-19,96836,"(actual, change, change_percent, consensus, co...",2026-06-20T16:02:45.172072+00:00
5,RISK_PREMIUM,macro_risk_premium,fmp,None,None,192,"(continent, country, country_risk_premium, tot...",2026-06-20T16:00:41.736420+00:00


## Read Examples

In [7]:
preview(warehouse.read_macro_panel(["GDP", "CPI"], provider=PROVIDER))

,GDP,CPI
date,,
2026-01-01,31819.464,326.588
2026-02-01,NaN,327.460
2026-03-01,NaN,330.293
2026-04-01,NaN,332.407
2026-05-01,NaN,333.979


In [8]:
preview(warehouse.macro.read_calendar(provider=PROVIDER))

,country,event,importance,currency,unit,previous,actual,change,change_percent,consensus
date,,,,,,,,,,
2026-06-19 13:30:00,SI,Unemployment Rate (Apr),Low,EUR,%,4.6,4.5,-0.1,-0.02174,4.6
2026-06-19 14:30:00,EU,ECB Lane Speech,Low,EUR,NaN,NaN,NaN,NaN,NaN,NaN
2026-06-19 15:00:00,RU,M2 Money Supply YoY (May),Low,RUB,NaN,12.3,13.0,0.7,0.05691,NaN
2026-06-19 19:00:00,AR,Retail Sales YoY (Apr),Low,ARS,%,3.6,12.6,9.0,2.50000,1.0
2026-06-19 19:30:00,US,CFTC S&P 500 speculative net positions,Medium,USD,K,-205.6,NaN,NaN,NaN,NaN


In [9]:
warehouse.macro.read_risk_premium(provider=PROVIDER)

,country,continent,total_equity_risk_premium,country_risk_premium
as_of,,,,
2026-06-20 00:00:00+00:00,Zimbabwe,Africa,15.89,11.66
2026-06-20 00:00:00+00:00,Zambia,Africa,15.89,11.66
2026-06-20 00:00:00+00:00,"Yemen, Republic",Asia,19.77,15.54
2026-06-20 00:00:00+00:00,Yemen,Asia,20.35,16.02
2026-06-20 00:00:00+00:00,Vietnam,Asia,8.13,3.90
...,...,...,...,...
2026-06-20 00:00:00+00:00,Anguilla,North America,12.44,8.11
2026-06-20 00:00:00+00:00,Angola,Africa,12.64,8.41
2026-06-20 00:00:00+00:00,Andorra (Principality of),Europe,6.30,2.07


<!-- quant-trader-eda -->
## Quant Trader EDA

Macro checks: growth/inflation changes, equity sensitivity to macro series, and yield-curve regime diagnostics.

In [10]:
macro_panel = warehouse.read_macro_panel(["GDP", "CPI", "unemploymentRate", "federalFunds"], provider=PROVIDER)
if macro_panel.empty:
    macro_changes = pd.DataFrame()
else:
    macro_numeric = macro_panel.apply(pd.to_numeric, errors="coerce")
    macro_changes = pd.concat(
        {
            "level": macro_numeric,
            "change_1": macro_numeric.diff(),
            "pct_change_1": macro_numeric.pct_change(fill_method=None),
        },
        axis=1,
    )
preview(macro_changes, rows=10)

level                                        change_1                                      pct_change_1                                        
                  GDP      CPI unemploymentRate federalFunds      GDP    CPI unemploymentRate federalFunds          GDP       CPI unemploymentRate federalFunds
date                                                                                                                                                           
2025-08-01        NaN  323.291              4.3         4.33      NaN  1.122              0.0         0.00          NaN  0.003483         0.000000     0.000000
2025-09-01        NaN  324.245              4.4         4.22      NaN  0.954              0.1        -0.11          NaN  0.002951         0.023256    -0.025404
2025-10-01  31422.526      NaN              NaN         4.09      NaN    NaN              NaN        -0.13          NaN       NaN              NaN    -0.030806
2025-11-01        NaN  325.063              4.5         3.88      NaN    NaN              NaN        -0.21          NaN       NaN              NaN    -0.051345
2025-12-01        NaN  326.031              4.4         3.72      NaN  0.968             -0.1        -0.16          NaN  0.002978        -0.022222    -0.041237
2026-01-01  31819.464  326.588              4.3         3.64      NaN  0.557             -0.1        -0.08          NaN  0.001708        -0.022727    -0.021505
2026-02-01        NaN  327.460              4.4         3.64      NaN  0.872              0.1         0.00          NaN  0.002670         0.023256     0.000000
2026-03-01        NaN  330.293              4.3         3.64      NaN  2.833             -0.1         0.00          NaN  0.008651        -0.022727     0.000000
2026-04-01        NaN  332.407              4.3         3.64      NaN  2.114              0.0         0.00          NaN  0.006400         0.000000     0.000000
2026-05-01        NaN  333.979              4.3         3.63      NaN  1.572              0.0        -0.01          NaN  0.004729         0.000000    -0.002747

In [11]:
spy_prices = warehouse.etf.read_prices("SPY", provider=PROVIDER)
macro_panel = warehouse.read_macro_panel(["GDP", "CPI", "unemploymentRate", "federalFunds"], provider=PROVIDER)
if spy_prices.empty or macro_panel.empty:
    macro_equity_corr = pd.DataFrame()
else:
    monthly_spy = spy_prices["close"].resample("ME").last().pct_change().rename("SPY_monthly_return")
    monthly_macro = macro_panel.apply(pd.to_numeric, errors="coerce").resample("ME").last().ffill().diff()
    joined = pd.concat([monthly_spy, monthly_macro], axis=1).dropna()
    macro_equity_corr = joined.corr()[["SPY_monthly_return"]].drop(index="SPY_monthly_return", errors="ignore")
macro_equity_corr

,SPY_monthly_return
GDP,0.043728
CPI,0.031740
unemploymentRate,0.072893
federalFunds,0.055548


In [12]:
treasury_codes = warehouse.macro.list_treasury_series_codes(provider=PROVIDER)
treasury_codes[:20]

['macro__ust_month1',
 'macro__ust_month2',
 'macro__ust_month3',
 'macro__ust_month6',
 'macro__ust_year1',
 'macro__ust_year10',
 'macro__ust_year2',
 'macro__ust_year20',
 'macro__ust_year3',
 'macro__ust_year30',
 'macro__ust_year5',
 'macro__ust_year7']

In [13]:
if not treasury_codes:
    pd.DataFrame()
else:
    treasury_panel = pd.concat(
        {code: warehouse.macro.read_series(code, provider=PROVIDER) for code in treasury_codes[:20]},
        axis=1,
    ).dropna(how="all")
    preview(treasury_panel, rows=10)

<!-- quant-trader-signal-eda -->
## Signal Research for P&L

Macro data is usually useful when converted into regimes. These cells compare equity returns across inflation, policy-rate, unemployment, and yield-curve states.

In [14]:
spy_prices = warehouse.etf.read_prices("SPY", provider=PROVIDER)
macro_panel = warehouse.read_macro_panel(["CPI", "unemploymentRate", "federalFunds"], provider=PROVIDER)
if spy_prices.empty or macro_panel.empty:
    pd.DataFrame()
else:
    spy_monthly = spy_prices["close"].resample("ME").last().pct_change().rename("spy_return_fwd_1m").shift(-1)
    macro_monthly = macro_panel.apply(pd.to_numeric, errors="coerce").resample("ME").last().ffill()
    regimes = pd.DataFrame(index=macro_monthly.index)
    if "CPI" in macro_monthly.columns:
        regimes["cpi_rising"] = macro_monthly["CPI"].diff() > 0
    if "unemploymentRate" in macro_monthly.columns:
        regimes["unemployment_rising"] = macro_monthly["unemploymentRate"].diff() > 0
    if "federalFunds" in macro_monthly.columns:
        regimes["fed_funds_rising"] = macro_monthly["federalFunds"].diff() > 0
    joined = pd.concat([spy_monthly, regimes], axis=1).dropna()
    rows = []
    for regime_column in regimes.columns:
        grouped = joined.groupby(regime_column)["spy_return_fwd_1m"].agg(["count", "mean", "std"])
        grouped["annualized_mean"] = grouped["mean"] * 12
        grouped["annualized_vol"] = grouped["std"] * 12 ** 0.5
        grouped["regime"] = regime_column
        rows.append(grouped.reset_index().rename(columns={regime_column: "regime_value"}))
    pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

In [15]:
treasury_codes = warehouse.macro.list_treasury_series_codes(provider=PROVIDER)
if not treasury_codes:
    pd.DataFrame()
else:
    treasury_panel = pd.concat(
        {code: warehouse.macro.read_series(code, provider=PROVIDER) for code in treasury_codes},
        axis=1,
    ).dropna(how="all")
    candidate_10y = next((column for column in treasury_panel.columns if "10" in column), None)
    candidate_2y = next((column for column in treasury_panel.columns if "2" in column), None)
    if candidate_10y is None or candidate_2y is None:
        preview(treasury_panel, rows=10)
    else:
        curve = (treasury_panel[candidate_10y] - treasury_panel[candidate_2y]).rename("curve_10y_2y")
        curve_state = pd.DataFrame({
            "curve_10y_2y": curve,
            "curve_change_21d": curve.diff(21),
            "inverted": curve < 0,
        })
        preview(curve_state, rows=15)

<!-- output-analysis -->
## Analysis Notes From This Run

In the current stored macro sample, the simple one-month-forward `SPY` regime split does not show a large penalty from rising CPI, rising unemployment, or rising Fed funds. Annualized forward `SPY` returns were similar across the simple rising/falling policy-rate split, and even slightly higher in the rising CPI/unemployment buckets in this crude diagnostic.

This does not mean macro is irrelevant. It means these raw level-change flags are too blunt to trade directly. For money-making research, the next step is to test surprises, z-scores, yield-curve slope/inversion, inflation acceleration, and interactions with market trend/volatility rather than using simple up/down macro changes alone.